# 03 · ORB vs AKAZE (vs SIFT)

Feature matching should survive the rotation and scale that broke template matching. Compare detectors on the axes that matter for navigation: accuracy, robustness, and runtime.

**OpenCV 5 note:** `cv2.ORB` and `cv2.SIFT` are top-level, but AKAZE moved to `cv2.xfeatures2d.AKAZE`. Matcher norm must suit the descriptor — `NORM_HAMMING` for binary (ORB/AKAZE/BRISK), `NORM_L2` for float (SIFT).

In [ ]:
%load_ext autoreload
%autoreload 2

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from navmod import matcher, camera, metrics, georef

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"


def show(img, title=None, figsize=(11, 7)):
    """Inline display for OpenCV arrays (handles BGR->RGB and grayscale)."""
    plt.figure(figsize=figsize)
    gray = img.ndim == 2
    plt.imshow(img if gray else cv2.cvtColor(img, cv2.COLOR_BGR2RGB),
               cmap="gray" if gray else None)
    if title:
        plt.title(title)
    plt.axis("off")
    plt.show()


print("ready ·", cv2.__version__)

## One honest comparison harness

Same frames, same truth, swap only the detector. Worth defining the detector list once and looping, rather than copy-pasting three blocks that drift apart.

## Match and estimate

Lowe ratio test on `knnMatch`, then `cv2.estimateAffinePartial2D` with RANSAC. Resist `findHomography` here — see the note in `src/matching.py`.

## Score them

Per detector, over many random (x, y, heading, scale) samples: median error in metres, inlier ratio, failure rate, and time per frame. Medians over a sweep, not single frames — individual frames vary wildly by terrain.

## Decide, and say why

The winner depends on your constraint. ORB is fast and cheap (matters on real drone hardware), SIFT is usually most accurate, AKAZE often sits between. Write down the tradeoff you picked — notebook 04 inherits this choice.